<a href="https://colab.research.google.com/github/SofiaBarbas/TP-Investigacion-Operativa_Grupo1/blob/main/TP_IO_G1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python y librerías para el manejo y gráfico de ecuaciones
**Investigación Operativa – UTN.BA | Comisión K4571 | Grupo 1 | Tema 1**

Docente: Mg. Ing. Andrea Zumino

Integrantes: María del Rosario Cabaleiro, Gonzalo Duarte, Emily Higa, Sofía Barbás, Juan Francisco Ferchero

Caso práctico: Taller metalúrgico (Ejercicio 1.1 de la Guía Práctica).

In [ ]:
# Tema 1: Python y librerías (NumPy, Pandas, Matplotlib)
# Utilizamos también la librería SciPy para utilizar la función linprog
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

print("✅ Librerías cargadas exitosamente.")

## 1. Formulación del modelo

- x1 = piezas A por semana, x2 = piezas B por semana
- Max Z = 4·x1 + 3·x2
- Estampado: 6·x1 + 16·x2 ≤ 48.000 seg/semana
- Soldado: 12·x1 + 6·x2 ≤ 42.000 seg/semana
- Pintado: 9·x1 + 9·x2 ≤ 36.000 seg/semana
- x1, x2 ≥ 0

## 2. Resolución algorítmica (SciPy)

`linprog` tiene la particularidad de que siempre **minimiza**: para maximizar Z se minimiza −Z, por eso los coeficientes de `c` van con signo negativo.
Los argumentos se pasan **con nombre** (`A_ub=`, `b_ub=`, `bounds=`): el cuarto argumento posicional de `linprog` es `A_eq`, no `bounds`.

In [ ]:
# Minimización del negativo para maximizar Z = 4*x1 + 3*x2
c = [-4, -3]

# Matriz de coeficientes A y vector de disponibilidades b (seg/semana)
A = np.array([
    [6, 16],   # Estampado: 6x1 + 16x2 <= 48000
    [12, 6],   # Soldado:   12x1 + 6x2 <= 42000
    [9, 9]     # Pintado:   9x1 + 9x2  <= 36000
])
b = np.array([48000, 42000, 36000])

# Resolver con SciPy linprog (solver HiGHS)
res = linprog(c, A_ub=A, b_ub=b, bounds=(0, None), method='highs')

x1_opt, x2_opt = res.x[0], res.x[1]
z_opt = -res.fun   # se vuelve a cambiar el signo

print("✨ SOLUCIÓN ÓPTIMA ENCONTRADA:")
print(f" 🔹 Piezas A (x1*): {x1_opt:,.0f} unidades/semana")
print(f" 🔹 Piezas B (x2*): {x2_opt:,.0f} unidades/semana")
print(f" 💰 Utilidad Máxima (Z*): ${z_opt:,.0f}")

## 3. Tabla de holguras, recursos y precios sombra (Pandas)

- **Holgura**: tiempo que sobra de cada proceso en el óptimo (`res.slack`).
- **Precio sombra**: cuánto aumenta Z por cada segundo extra de capacidad (`-res.ineqlin.marginals`; se cambia el signo porque el modelo se resolvió minimizando −Z).

In [ ]:
holguras = res.slack
precios_sombra = -res.ineqlin.marginals
tiempo_usado = A @ res.x   # producto matriz-vector con NumPy

# Uso de np.where para clasificar cada restricción
estados = np.where(holguras > 1e-5, 'No Vinculante (Sobrante)', 'VINCULANTE (Agotado)')

df_recursos = pd.DataFrame({
    'Proceso / Restricción': ['Estampado', 'Soldado', 'Pintado'],
    'Tiempo Disponible (seg)': b,
    'Tiempo Utilizado (seg)': tiempo_usado.round(2),
    'Holgura (seg)': holguras.round(2),
    'Precio Sombra ($/seg)': precios_sombra.round(4),
    'Estado del Recurso': estados
})

print("📊 Análisis de Recursos:")
df_recursos

## 4. Resolución gráfica (Matplotlib)

Se despeja x2 de cada restricción tomada como igualdad, se sombrea la región factible (debajo de la recta más restrictiva) y se marca el óptimo.
La recta punteada es la función objetivo en Z* = 15.000: el último punto de la región que toca es el óptimo.

In [ ]:
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.figure(figsize=(10, 7), dpi=120)
x1_vals = np.linspace(0, 5000, 1000)

# Ecuaciones explícitas de las 3 rectas
y1 = (48000 - 6 * x1_vals) / 16.0   # Estampado
y2 = (42000 - 12 * x1_vals) / 6.0   # Soldado
y3 = (36000 - 9 * x1_vals) / 9.0    # Pintado

plt.plot(x1_vals, y1, label='Estampado: 6x1 + 16x2 ≤ 48.000', color='blue', linewidth=2)
plt.plot(x1_vals, y2, label='Soldado: 12x1 + 6x2 ≤ 42.000', color='red', linewidth=2)
plt.plot(x1_vals, y3, label='Pintado: 9x1 + 9x2 ≤ 36.000', color='green', linewidth=2)

# Región factible: debajo de la recta más restrictiva y por encima de x2 = 0
y_factible = np.maximum(0, np.minimum(y1, np.minimum(y2, y3)))
plt.fill_between(x1_vals, 0, y_factible, color='lightgreen', alpha=0.4, label='Región Factible')

# Función objetivo evaluada en el óptimo: 4x1 + 3x2 = Z*
y_obj = (z_opt - 4 * x1_vals) / 3
plt.plot(x1_vals[y_obj >= 0], y_obj[y_obj >= 0], '--', color='purple', linewidth=2,
         label=f'Función objetivo: 4x1 + 3x2 = {z_opt:,.0f}'.replace(',', '.'))

# Punto óptimo
plt.plot(x1_opt, x2_opt, 'o', color='gold', markeredgecolor='black', markersize=12,
         label='Óptimo (3.000; 1.000) | Z* = $15.000')
plt.annotate('  Punto Óptimo (3.000; 1.000)\n  Z* = $15.000',
             xy=(x1_opt, x2_opt), xytext=(x1_opt + 300, x2_opt + 1300),
             arrowprops=dict(facecolor='black', shrink=0.08, width=1.5, headwidth=8),
             fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='yellow', alpha=0.8))

plt.xlim(0, 4500)
plt.ylim(0, 4500)
plt.xlabel('Piezas A (x1)', fontsize=12)
plt.ylabel('Piezas B (x2)', fontsize=12)
plt.title('Método Gráfico - Taller Metalúrgico (Grupo 1 K4571)', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.savefig('grafico_metodo_grafico.png', dpi=200)
plt.show()

## 5. Verificación: evaluación de Z en los vértices

Teorema fundamental de la PL: si existe óptimo, está en un vértice de la región factible.

In [ ]:
vertices = {
    '(0; 0) – origen': (0, 0),
    '(3.500; 0) – Soldado y eje x1': (3500, 0),
    '(3.000; 1.000) – Soldado y Pintado': (3000, 1000),
    '(1.600; 2.400) – Pintado y Estampado': (1600, 2400),
    '(0; 3.000) – Estampado y eje x2': (0, 3000),
}
df_vertices = pd.DataFrame(
    [(nombre, 4 * x1 + 3 * x2) for nombre, (x1, x2) in vertices.items()],
    columns=['Vértice', 'Z = 4x1 + 3x2']
).sort_values('Z = 4x1 + 3x2', ascending=False)
df_vertices

## 6. Análisis de sensibilidad

El análisis de sensibilidad responde **qué pasa si cambian los datos** sin volver a plantear todo el problema:

- **Recursos (lado derecho, b):** cuánto vale un segundo extra de cada proceso (precio sombra) y hasta dónde ese valor sigue siendo válido.
- **Utilidades (coeficientes de Z, c):** cuánto puede variar la ganancia de cada pieza sin que cambie el plan óptimo.

La solución óptima queda definida por las **restricciones activas** (Soldado y Pintado). Con NumPy armamos esa matriz base `B` y calculamos cómo se mueve el vértice óptimo cuando cambia cada disponibilidad.

In [ ]:
activas = np.where(res.slack < 1e-5)[0]          # índices de las restricciones agotadas
nombres = np.array(['Estampado', 'Soldado', 'Pintado'])
B = A[activas]                                     # matriz 2x2 de las restricciones activas
B_inv = np.linalg.inv(B)
print('Restricciones activas:', ', '.join(nombres[activas]))

def rango_recurso(i):
    """Rango de b_i en el que el precio sombra sigue siendo válido."""
    if i not in activas:                           # recurso con holgura: alcanza con no bajar de lo usado
        return tiempo_usado[i], np.inf
    k = list(activas).index(i)
    d = B_inv[:, k]                                # cómo se mueve (x1, x2) por cada segundo extra de b_i
    t_min, t_max = -np.inf, np.inf
    # 1) x1 y x2 deben seguir siendo >= 0
    for xj, dj in zip(res.x, d):
        if dj > 0: t_min = max(t_min, -xj / dj)
        if dj < 0: t_max = min(t_max, -xj / dj)
    # 2) las restricciones no activas deben seguir cumpliéndose
    for j in range(len(b)):
        if j in activas: continue
        uso = A[j] @ d
        if uso > 0: t_max = min(t_max, res.slack[j] / uso)
        if uso < 0: t_min = max(t_min, res.slack[j] / uso)
    return b[i] + t_min, b[i] + t_max

rangos = [rango_recurso(i) for i in range(len(b))]
df_sens_b = pd.DataFrame({
    'Recurso': nombres,
    'Disponible (seg)': b,
    'Precio sombra ($/seg)': precios_sombra.round(4),
    '1 hora extra ($/semana)': (precios_sombra * 3600).round(0),
    'Rango mínimo (seg)': [round(r[0], 0) for r in rangos],
    'Rango máximo (seg)': [round(r[1], 0) if np.isfinite(r[1]) else '∞' for r in rangos],
})
print('📊 Sensibilidad de los recursos:')
df_sens_b

### Rango de las utilidades

El vértice (3.000; 1.000) sigue siendo óptimo mientras la pendiente de la función objetivo (c1 / c2) esté entre las pendientes de las dos restricciones activas: Pintado (9/9 = 1) y Soldado (12/6 = 2).

In [ ]:
pendientes = A[activas, 0] / A[activas, 1]     # pendiente a1/a2 de cada restricción activa
p_min, p_max = pendientes.min(), pendientes.max()
c1, c2 = 4, 3

df_sens_c = pd.DataFrame({
    'Coeficiente': ['c1 (utilidad pieza A)', 'c2 (utilidad pieza B)'],
    'Valor actual ($)': [c1, c2],
    'Mínimo ($)': [p_min * c2, c1 / p_max],
    'Máximo ($)': [p_max * c2, c1 / p_min],
})
print(f'Pendientes de las restricciones activas: {p_min:.0f} y {p_max:.0f}')
df_sens_c

### Verificación: volver a resolver con los datos cambiados

Para comprobar los rangos, volvemos a llamar a `linprog` con valores apenas dentro y apenas fuera de cada límite.

- Con la utilidad de A fuera de \$3 – \$6, el plan óptimo salta a otro vértice.
- Con más tiempo de Pintado cambian las cantidades, pero siguen activas las mismas restricciones; pasados los 40.846 seg, Z deja de crecer (\$16.076,9) porque el segundo extra ya no se aprovecha.

In [ ]:
def resolver(c=(4, 3), b_nuevo=b):
    r = linprog([-c[0], -c[1]], A_ub=A, b_ub=b_nuevo, bounds=(0, None), method='highs')
    return r.x.round(1), round(-r.fun, 1)

pruebas = [
    ('Utilidad A = $3,1 (dentro)', (3.1, 3), b),
    ('Utilidad A = $2,9 (fuera)',  (2.9, 3), b),
    ('Utilidad A = $5,9 (dentro)', (5.9, 3), b),
    ('Utilidad A = $6,1 (fuera)',  (6.1, 3), b),
    ('Pintado = 40.800 seg (dentro)', (4, 3), np.array([48000, 42000, 40800])),
    ('Pintado = 41.000 seg (fuera)',  (4, 3), np.array([48000, 42000, 41000])),
]
filas = []
for nombre, c_p, b_p in pruebas:
    x, z = resolver(c_p, b_p)
    filas.append((nombre, f'({x[0]:,.0f}; {x[1]:,.0f})'.replace(',', '.'), z))
pd.DataFrame(filas, columns=['Escenario', 'Plan óptimo (x1; x2)', 'Z* ($)'])

### Gráfico: utilidad máxima según el tiempo disponible de Pintado

Resolvemos el modelo para muchos valores de disponibilidad de Pintado. Dentro del rango de validez, Z crece con pendiente constante (el precio sombra, \$0,22 por segundo); fuera del rango la pendiente cambia porque cambia el vértice óptimo.

In [ ]:
b3_vals = np.linspace(26000, 46000, 201)
z_vals = [resolver(b_nuevo=np.array([48000, 42000, v]))[1] for v in b3_vals]
lo, hi = rangos[2]

plt.figure(figsize=(10, 5.5), dpi=120)
plt.plot(b3_vals, z_vals, color='green', linewidth=2.5, label='Z* según disponibilidad de Pintado')
plt.axvspan(lo, hi, color='lightgreen', alpha=0.35, label=f'Rango de validez: {lo:,.0f} a {hi:,.0f} seg'.replace(',', '.'))
plt.plot(36000, z_opt, 'o', color='gold', markeredgecolor='black', markersize=11, label='Situación actual (36.000 seg; Z* = $15.000)')
plt.xlabel('Tiempo disponible de Pintado (seg/semana)', fontsize=12)
plt.ylabel('Utilidad máxima Z* ($/semana)', fontsize=12)
plt.title('Sensibilidad de Z* a la disponibilidad de Pintado', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

## 7. Conclusión

- Plan óptimo: **3.000 piezas A y 1.000 piezas B por semana**, utilidad **\$15.000**.
- Soldado y Pintado se agotan (cuellos de botella); Estampado tiene 14.000 seg de holgura.
- **Sensibilidad de recursos:** una hora extra de Pintado suma \$800 por semana (válido entre 31.500 y 40.846 seg) y una de Soldado \$600 (entre 33.600 y 48.000 seg). Ampliar Estampado no agrega utilidad.
- **Sensibilidad de utilidades:** el plan no cambia mientras la utilidad de A esté entre \$3 y \$6, y la de B entre \$2 y \$4. Dentro de esos rangos solo cambia la ganancia, no cuánto producir.